# Extração de Características Multimodais de cfDNA em Promotores


---

## O que este notebook faz

Ele transforma arquivos de coordenadas de fragmentos de DNA em quatro representações numéricas diferentes do mesmo paciente, treina um classificador sobre cada uma, e as compara entre si numa coorte que nunca foi vista durante o desenvolvimento.

O produto final são dez arquivos em `02_features/`, que alimentam tanto o classificador deste notebook quanto o estudo de benchmark que o sucede.

---

## O problema, em termos simples

No plasma circulam milhões de fragmentos curtos de DNA livre — o **cfDNA** — liberados por células que morreram em qualquer tecido. Quando há um tumor, uma fração desses fragmentos veio dele. Em doença inicial, essa fração fica abaixo de 1%.

Fragmentos de origem tumoral têm propriedades físicas distintas: são em média mais curtos, terminam em posições diferentes do genoma e se distribuem de maneira desigual pelos cromossomos. A **fragmentômica** explora essas diferenças.

A pergunta deste notebook é qual descrição de um fragmento carrega mais sinal diagnóstico: a sequência lida por um modelo de fundação genômico, a contagem de combinações de bases, a posição do promotor atingido, o comprimento da molécula, ou a cobertura agregada em janelas grandes do genoma.

---

## As duas geografias

Entender este ponto torna o resto do notebook legível.

**O painel** é a região do genoma que sobreviveu ao recorte feito antes do upload. São as vizinhanças de ±2 kb do centro de cada promotor humano catalogado, fundidas quando próximas. Os arquivos de fragmento no disco contêm **apenas** fragmentos que tocam essa região — o recorte reduziu 280 GB para cerca de 12 GB.

**A janela promotora** é mais estreita: ±1 kb do centro de um promotor. É ela que define quais fragmentos entram na bolsa do paciente.

Um fragmento pertence a uma região se o seu **ponto médio** cai dentro dela. Usar o ponto médio, e não a sobreposição, evita que uma molécula seja contada em duas regiões vizinhas.

```
painel           ├────────────── ±2 kb ──────────────┤   o que está no disco
janela promotora      ├───── ±1 kb ─────┤               o que entra na bolsa
centro do promotor              ▲
```

---

## O caminho dos dados

```
arquivos de fragmento (coordenadas)
        │
        ├─ CÉLULA 03 ── conta fragmentos elegíveis por paciente
        │               decide quem tem os 5.000 necessários
        │
        ├─ CÉLULA 05 ── UMA passada por arquivo, produzindo de uma vez:
        │               · amostra de 5.000 fragmentos
        │               · histograma de comprimentos
        │               · três contagens de cobertura em 5 Mb
        │
        │               para cada um dos 5.000:
        │               · lê a sequência no hg38
        │               · conta os 256 tetranucleotídeos
        │               · passa pelo Caduceus
        │
        ├─ CÉLULA 07 ── treina e congela os modelos, só em Baltimore
        │
        └─ CÉLULA 08 ── avalia em Hong Kong, com os pesos travados
```

---

## As quatro representações comparadas

Cada paciente vira uma **bolsa** de 5.000 fragmentos. O que muda entre os braços é o que cada fragmento carrega dentro dela.

| Braço         | O que descreve cada fragmento                           | Dim. |
| :------------ | :------------------------------------------------------ | ---: |
| `caduceus`    | o que o modelo de fundação «vê» na sequência ao redor   |  256 |
| `kmer`        | com que frequência cada combinação de 4 letras aparece  |  256 |
| `promoter_id` | qual promotor o fragmento atingiu, como vetor aprendido |  256 |
| `length_only` | nada — só o canal de comprimento sobrevive              |    0 |

Os quatro compartilham **tudo o mais**: a mesma bolsa, a mesma arquitetura, os mesmos hiperparâmetros, as mesmas partições. É isso que torna a comparação interpretável — a única coisa que difere é a informação que entra.

O braço `length_only` é o controle de canal único: mede quanto se obtém apenas com o tamanho das moléculas, sem nenhuma informação de sequência ou posição.

---

## Por que três versões do DELFI

O DELFI clássico conta fragmentos curtos e longos em janelas de cinco milhões de bases. Aqui ele é construído três vezes, de propósito:

| Arquivo                      | Fragmentos usados                 | Serve para                               |
| :--------------------------- | :-------------------------------- | :--------------------------------------- |
| `delfi_all.npy`              | todos com ponto médio no painel   | a versão mais próxima do DELFI original  |
| `delfi_promoter_all.npy`     | todos com ponto médio em promotor | isola o efeito da restrição a promotores |
| `delfi_promoter_matched.npy` | **exatamente os 5.000 da bolsa**  | isola o efeito da profundidade           |

O terceiro é o mais importante para a comparação. Quando o DELFI vence um braço neural, há duas explicações possíveis: a cobertura em megabases é intrinsecamente mais robusta, ou o DELFI apenas viu duzentas vezes mais moléculas. O `delfi_promoter_matched` usa as mesmas 5.000 e separa as duas.

---

## A arquitetura

```
5.000 fragmentos × 256 números
        │
    LayerNorm
        │
canal de comprimento
        │
Projeção (Linear → ReLU → Dropout) ← 256+1 para 128
        │
Atenção com porta (tanh × sigmoid) ← um peso por fragmento
        │
Soma ponderada ← a bolsa vira um vetor
        │
histograma de comprimentos da bolsa ← multimodalidade
        │
Classificador
```

O **Multiple Instance Learning** existe porque sabemos o diagnóstico do paciente, mas não quais moléculas vieram do tumor. A bolsa tem rótulo; as instâncias não. A atenção com porta aprende um peso para cada fragmento, com a promessa de destacar os informativos.

O **histograma** entra por fora da atenção. Ele resume o comprimento de todos os fragmentos promotores do paciente, não apenas os 5.000 amostrados, e dá ao classificador uma visão da bolsa inteira.

---

## O que impede o resultado de ser acidente

Estes são os mecanismos que o notebook usa para que um resultado bom não possa ser fruto de escolha posterior.

**Assinatura do protocolo.** A configuração científica inteira é reduzida a um código SHA-256. Mudar qualquer parâmetro muda o código, e todos os arquivos intermediários de execuções anteriores deixam de ser aceitos. Campos que controlam apenas a execução — como em que etapa parar — ficam de fora da assinatura, para que ligá-los ou desligá-los não invalide o protocolo.

**Portões baratos primeiro.** A Célula 01 verifica GPU, arquivos obrigatórios e a impressão digital MD5 do painel antes de qualquer processamento caro. Falhar em quinze segundos é melhor que falhar depois de uma hora de embeddings.

**Amostragem por reservatório.** Os 5.000 fragmentos são sorteados uniformemente entre todos os elegíveis, numa única passada pelo arquivo e sem saber de antemão quantos existem. A semente do sorteio deriva do identificador da amostra, então o mesmo paciente produz sempre a mesma bolsa.

**Tamanho da bolsa escolhido por retenção.** A Célula 03 mostra quantos pacientes sobrevivem a bolsas de 2.000, 5.000 e 10.000 fragmentos. O valor foi fixado olhando quantos pacientes se perdem, não olhando qual dá melhor resultado.

**Dobras congeladas.** As cinco partições de Baltimore são gravadas em disco antes de qualquer treino e reutilizadas por todos os braços.

**Hong Kong só depois do congelamento.** A Célula 07 termina gravando a assinatura SHA-256 de cada peso treinado. A Célula 08 só então abre a coorte externa. Os rótulos de Hong Kong não influenciam nenhuma decisão.

**Regra de decisão anterior ao resultado.** O limiar de relevância está fixado em 0,03 de AUROC. A conclusão é escolhida por uma regra escrita antes de ver os números, e a futilidade tem precedência:

| Condição                                         | Veredito                      |
| :----------------------------------------------- | :---------------------------- |
| limite superior do intervalo abaixo de 0,03      | `FUTIL`                       |
| diferença ≥ 0,03 e limite inferior acima de zero | `SUPERIOR_PROMISSOR`          |
| diferença ≥ 0,03 sem o limite inferior           | `PROMISSOR_REQUER_REPLICACAO` |
| demais casos                                     | `INCONCLUSIVO`                |

---

## Parâmetros que definem o estudo

| Parâmetro               |     Valor | O que significa                               |
| :---------------------- | --------: | :-------------------------------------------- |
| `MAPQ`                  |        30 | qualidade mínima de mapeamento                |
| `FRAG_MIN` / `FRAG_MAX` | 100 / 220 | faixa de comprimento aceita, em pares de base |
| `SHORT_MAX`             |       150 | limite entre «curto» e «longo» para o DELFI   |
| `FLANK_PRIMARY`         |         6 | bases lidas além de cada ponta do fragmento   |
| `PROMOTER_HALF_WINDOW`  |     1.000 | meia-largura da janela promotora              |
| `PROMOTER_POOL`         |     5.000 | fragmentos por paciente                       |
| `BIN_SIZE`              | 5.000.000 | largura da janela do DELFI                    |
| `CADUCEUS_LAYER`        |        −1 | camada final do encoder                       |
| `N_FOLDS` / `N_SEEDS`   |     5 / 5 | partições e repetições                        |
| `DELTA_STAR`            |      0,03 | diferença mínima considerada relevante        |

Com fragmento mediano de 166 pares de base e margem de ±6, a janela lida pelo Caduceus tem cerca de **178 pares de base**. O embedding de cada fragmento é a **média da camada final sobre as posições dessa janela** — ou seja, a ordem interna é colapsada antes de qualquer agregação entre fragmentos.

Fragmentos mapeados na fita negativa são revertidos e complementados antes da codificação. O Caduceus é equivariante a reverso-complemento por desenho.

---

## O que é gravado

**`00_lock/`** — tudo o que precisa existir antes do primeiro treino

| Arquivo                                         | Conteúdo                                       |
| :---------------------------------------------- | :--------------------------------------------- |
| `protocol_config.json`                          | a configuração e sua assinatura                |
| `promoter_catalog.csv`                          | os promotores usados, com identificador        |
| `promoter_panel_union.bed`                      | a interseção entre janelas promotoras e painel |
| `promoter_fragment_counts.csv`                  | fragmentos elegíveis por paciente              |
| `eligible_samples.csv` · `excluded_samples.csv` | quem entrou e quem saiu                        |
| `cristiano_folds.csv`                           | as cinco partições, congeladas                 |

**`02_features/`** — as representações, uma linha por paciente

| Arquivo                             | Forma          | Conteúdo                                         |
| :---------------------------------- | :------------- | :----------------------------------------------- |
| `metadata.csv`                      | —              | **define a ordem das linhas de todos os demais** |
| `promoter_fragment_coordinates.npy` | (N, 5000, 4)   | cromossomo, início, fim, fita                    |
| `promoter_fragment_lengths.npy`     | (N, 5000)      | comprimento de cada fragmento                    |
| `promoter_ids.npy`                  | (N, 5000)      | promotor atingido                                |
| `caduceus_embeddings.npy`           | (N, 5000, 256) | embedding por fragmento                          |
| `kmer_features.npy`                 | (N, 5000, 256) | frequência de tetranucleotídeos                  |
| `promoter_histograms.npy`           | (N, 30)        | histograma de comprimentos da bolsa              |
| `delfi_all.npy`                     | (N, 3×bins)    | cobertura, todos os fragmentos do painel         |
| `delfi_promoter_all.npy`            | (N, 3×bins)    | cobertura, todos os promotores                   |
| `delfi_promoter_matched.npy`        | (N, 3×bins)    | cobertura, apenas os 5.000                       |

**`03_models/`** — pesos por dobra e por semente, com assinatura  
**`04_results/`** — AUROC por braço, contrastes pareados, curvas, relatório

> **Os arquivos `.npy` não carregam identificador de paciente.** A linha `i` de qualquer um deles corresponde à linha `i` de `metadata.csv`, e essa é a única garantia de pareamento. Qualquer notebook que consuma estas saídas precisa preservar essa ordem.

---

## Como executar

O notebook tem três estágios, controlados por `RUN_STAGE`:

| Valor        | Para onde vai                                       |
| :----------- | :-------------------------------------------------- |
| `AUDIT`      | para após contar fragmentos e definir elegibilidade |
| `FEATURES`   | para após gravar as representações                  |
| `TRAIN_EVAL` | treina e avalia, assumindo features prontas         |
| `ALL`        | tudo de uma vez                                     |

A extração dos embeddings é a etapa cara. Interromper em `FEATURES`, publicar `caduceus_promoter_mil_v3` como dataset, e retomar depois evita refazê-la: a Célula 01 procura um `protocol_config.json` compatível nos datasets anexados e copia o que já existe.

**Configuração no Kaggle**

- Accelerator: GPU T4 ou P100 (obrigatória)
- Internet: On na primeira execução
- Persistence: Files only

**Datasets necessários**

| Dataset                               | Conteúdo                                        |
| :------------------------------------ | :---------------------------------------------- |
| `cfdna-ref-hg38`                      | `hg38.2bit`, `GRCh38-PLS.bed`, `painel_ext.bed` |
| `cfdna-cristiano2019-delfi`           | `manifest.csv` e fragmentos recortados          |
| `cfdna-jiang2015-hcc`                 | `manifest.csv` e fragmentos recortados          |
| `caduceus-ph-vanilla`                 | checkpoint do encoder                           |
| `cfdna-promoter-multimodal-resume-v3` | opcional, para retomar                          |

---

## Duas ressalvas sobre esta versão

**O controle técnico não é executado nesta revisão.** Na Célula 04, o arquivo `technical_gate.json` é gravado com `passed=True` e um valor herdado de execução anterior, sem refazer o teste. A versão 2 implementava o teste completo: treinava uma sonda de Poisson para prever densidade de clivagem a partir dos embeddings, comparava contra embeddings embaralhados e exigia um intervalo de confiança acima de zero. Quem precisar dessa garantia deve restaurar aquela implementação.

**As sensibilidades pós-lock estão desligadas.** A Célula 09 é um esqueleto. `RUN_POSTLOCK_SENSITIVITIES=True` não executa nada nesta revisão.

In [ ]:
# >>> CELL 00 START — CABECALHO, CONFIGURACAO E DIRETORIOS ====================
print("\n" + "=" * 90)
print("BEGIN CELL 00 — CABECALHO, CONFIGURACAO E DIRETORIOS")
print("=" * 90)

import copy
import gc
import glob
import hashlib
from importlib.metadata import PackageNotFoundError, version
import importlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import subprocess
import sys
import time
import warnings
import zipfile

ROOT = Path("/kaggle/working/caduceus_promoter_mil_v3")
DIRS = {
    "lock": ROOT / "00_lock",
    "gate": ROOT / "01_technical_gate",
    "features": ROOT / "02_features",
    "models": ROOT / "03_models",
    "results": ROOT / "04_results",
    "sens": ROOT / "05_sensitivities",
    "logs": ROOT / "logs",
}
for _d in DIRS.values():
  _d.mkdir(parents=True, exist_ok=True)

NOTEBOOK_REVISION = "3.1.0"

CFG = dict(
    STUDY_VERSION="3.1.0",
    PANEL_MD5="1dc132ed3e9fa79596495ebf30ab63f0",
    SOURCE_STUDY="cristiano2019_delfi_wgs",
    TARGET_STUDY="jiang2015_hcc_wgs",
    HEALTHY_LABEL="healthy",
    CANCER_LABEL="liver_cancer",
    LIVER_DISEASE_LABELS=("hepatitis_b", "cirrhosis"),
    # Fragmentos e janelas promotoras
    MAPQ=30,
    FRAG_MIN=100,
    FRAG_MAX=220,
    SHORT_MAX=150,
    FLANK_PRIMARY=6,
    PROMOTER_HALF_WINDOW=1_000,
    PROMOTER_POOL=5_000,
    PROMOTER_TRAIN_BAG=5_000,
    POOL_AUDIT_CANDIDATES=(2_000, 5_000, 10_000),
    MIN_COHORT_RETENTION=0.90,
    BIN_SIZE=5_000_000,
    MIN_PANEL_BP_PER_BIN=20_000,
    MIN_PROMOTER_BP_PER_BIN=2_000,
    N_GC_DECILES=10,
    # Multimodalidade e Histograma de bolsa
    USE_HISTOGRAM=True,
    HIST_RANGE=(70, 250),
    N_HIST_BINS=30,
    # Caduceus
    CADUCEUS_LAYER=-1,
    CAD_BATCH=128,
    CAD_DTYPE="float16",
    # Controle tecnico
    TECH_WIN=1_024,
    TECH_N_REGIONS=1_200,
    TECH_TRAIN_CHR=tuple(range(1, 15)),
    TECH_VALID_CHR=(15, 16),
    TECH_TEST_CHR=tuple(range(17, 23)),
    TECH_EPOCHS=100,
    TECH_PATIENCE=12,
    TECH_BATCH_REGIONS=8,
    TECH_PASS_DELTA=0.01,
    TECH_MAX_SHUFFLED=0.02,
    TECH_BOOT=500,
    TECH_BLOCK_SIZE=10_000_000,
    # MIL
    INSTANCE_DIM=128,
    ATTENTION_DIM=64,
    DROPOUT=0.25,
    LR=1e-3,
    WEIGHT_DECAY=1e-4,
    MAX_EPOCHS=35,
    PATIENCE=6,
    N_FOLDS=5,
    N_SEEDS=5,
    TRAIN_VAL_FRACTION=0.15,
    GRAD_CLIP=5.0,
    # Inferencia
    N_BOOT=2_000,
    DELTA_STAR=0.03,
    TARGET_SPECIFICITY=0.98,
    # Precisao
    PRECISION_SIMS=1_000,
    PRECISION_AUC_OFFSETS=(-0.05, 0.0, 0.05),
    PRECISION_DELTAS=(-0.05, 0.0, 0.03, 0.05, 0.10),
    PRECISION_CORR_PAIRS=(
        (0.3, 0.3),
        (0.5, 0.5),
        (0.7, 0.7),
        (0.9, 0.9),
        (0.3, 0.7),
        (0.7, 0.3),
    ),
    # Execucao: AUDIT -> FEATURES -> TRAIN_EVAL ou ALL
    RUN_STAGE="ALL",
    RUN_POSTLOCK_SENSITIVITIES=False,
    FORCE_REBUILD_FEATURES=False,
    FORCE_RETRAIN_MODELS=False,
    SEED=42,
)

PRIMARY_CFG = {
    k: v
    for k, v in CFG.items()
    if k
    not in {
        "RUN_STAGE",
        "RUN_POSTLOCK_SENSITIVITIES",
        "FORCE_REBUILD_FEATURES",
        "FORCE_RETRAIN_MODELS",
    }
}


def canonical_json(obj):
  return json.dumps(obj, sort_keys=True, separators=(",", ":"), default=list)


def sha256_bytes(data: bytes) -> str:
  return hashlib.sha256(data).hexdigest()


def sha256_file(path, block=1 << 20):
  h = hashlib.sha256()
  with open(path, "rb") as f:
    while True:
      b = f.read(block)
      if not b:
        break
      h.update(b)
  return h.hexdigest()


def md5_file(path, block=1 << 20):
  h = hashlib.md5()
  with open(path, "rb") as f:
    while True:
      b = f.read(block)
      if not b:
        break
      h.update(b)
  return h.hexdigest()


def hash_object(obj):
  return sha256_bytes(canonical_json(obj).encode())


PROTOCOL_HASH = hash_object(PRIMARY_CFG)
(DIRS["lock"] / "protocol_config.json").write_text(
    json.dumps(
        {"protocol_hash": PROTOCOL_HASH, "config": CFG}, indent=2, default=list
    )
)
print(f"Protocol hash    : {PROTOCOL_HASH[:16]}…")
print(f"Notebook revision: {NOTEBOOK_REVISION}")
if CFG["RUN_STAGE"] not in {"AUDIT", "FEATURES", "TRAIN_EVAL", "ALL"}:
  raise ValueError("RUN_STAGE deve ser AUDIT, FEATURES, TRAIN_EVAL ou ALL")
print(f"Working dir      : {ROOT}")
print(f"Run stage        : {CFG['RUN_STAGE']}")
print("=" * 90)
print("END CELL 00 — CABECALHO, CONFIGURACAO E DIRETORIOS")
print("=" * 90)
# <<< CELL 00 END =============================================================

In [ ]:
# >>> CELL 01 START — AMBIENTE E PORTOES BARATOS ==============================
print("\n" + "=" * 90)
print("BEGIN CELL 01 — AMBIENTE E PORTOES BARATOS")
print("=" * 90)

INPUT_ROOTS = [Path("/kaggle/input")]
TEMP_EXTRACT = Path("/kaggle/temp/cfdna")
TEMP_EXTRACT.mkdir(parents=True, exist_ok=True)


def find_all(pattern):
  out = []
  for base in INPUT_ROOTS:
    out.extend(
        Path(p) for p in glob.glob(str(base / "**" / pattern), recursive=True)
    )
  return sorted(set(out))


def attached_dataset_names():
  root = Path("/kaggle/input")
  return (
      sorted(p.name for p in root.iterdir() if p.is_dir())
      if root.exists()
      else []
  )


def find_one(name, required=True):
  xs = find_all(name)
  if required and not xs:
    raise FileNotFoundError(
        f"Arquivo obrigatório não encontrado: {name}. "
        f"Datasets anexados: {attached_dataset_names()}. "
        "Anexe cfdna-ref-hg38 ou confirme que o arquivo está dentro de um ZIP"
        " válido."
    )
  return xs[0] if xs else None


print(f"Datasets anexados: {attached_dataset_names()}")

_input_zips = find_all("*.zip")
for z in _input_zips:
  suffix = hashlib.sha256(str(z).encode()).hexdigest()[:8]
  dest = TEMP_EXTRACT / f"{z.stem}_{suffix}"
  marker = dest / ".done"
  if marker.exists():
    continue
  dest.mkdir(parents=True, exist_ok=True)
  print(f"Extraindo {z.name} …", flush=True)
  if not zipfile.is_zipfile(z):
    raise SystemExit(f"Arquivo com extensão ZIP inválido: {z}")
  with zipfile.ZipFile(z) as f:
    f.extractall(dest)
  marker.touch()
if _input_zips:
  INPUT_ROOTS.append(TEMP_EXTRACT)
  print(f"ZIPs extraídos: {len(_input_zips)} | raiz temporária: {TEMP_EXTRACT}")

_resume_markers = sorted(
    Path(p)
    for base in INPUT_ROOTS
    for p in glob.glob(
        str(base / "**/caduceus_promoter_mil_v3/00_lock/protocol_config.json"),
        recursive=True,
    )
)
if _resume_markers:
  resume_root = _resume_markers[0].parents[1]
  print(f"Checkpoint opcional encontrado: {resume_root}")
  for src in resume_root.rglob("*"):
    if not src.is_file():
      continue
    rel = src.relative_to(resume_root)
    dst = ROOT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
      shutil.copy2(src, dst)


def pkg_version(name):
  try:
    return version(name)
  except PackageNotFoundError:
    return None


def pip_install(spec):
  print(f"Instalando {spec} …", flush=True)
  r = subprocess.run(
      f"{sys.executable} -m pip install -q {spec}",
      shell=True,
      capture_output=True,
      text=True,
  )
  if r.returncode != 0:
    raise RuntimeError(r.stderr[-3000:])


_smi = subprocess.run(
    "nvidia-smi -L", shell=True, capture_output=True, text=True
)
if _smi.returncode != 0 or "GPU" not in _smi.stdout:
  raise SystemExit("SEM GPU — Kaggle Settings > Accelerator > GPU T4/P100")
print(_smi.stdout.strip().splitlines()[0])

HG38 = find_one("hg38.2bit")
PLS = find_one("GRCh38-PLS.bed")
PANEL = find_one("painel_ext.bed")
if md5_file(PANEL) != CFG["PANEL_MD5"]:
  raise SystemExit(f"painel_ext.bed divergente: {md5_file(PANEL)}")
print(f"Painel MD5 OK: {CFG['PANEL_MD5']}")

FRAGMENT_FILES = find_all("*.frag.tsv.bgz")
MANIFEST_FILES = find_all("manifest.csv")
if not FRAGMENT_FILES or len(MANIFEST_FILES) < 2:
  raise SystemExit("Anexe as duas coortes, com fragmentos e manifest.csv")
print(
    f"Fragment files: {len(FRAGMENT_FILES)} | manifests: {len(MANIFEST_FILES)}"
)

CAD_DIRS = []
for p in find_all("config.json"):
  if list(p.parent.glob("*.safetensors")):
    try:
      cfg = json.loads(p.read_text())
      if "caduceus" in str(cfg).lower() or "mamba" in str(cfg).lower():
        CAD_DIRS.append(p.parent)
    except Exception:
      pass
if not CAD_DIRS:
  raise SystemExit("Checkpoint Caduceus não encontrado")
CAD_DIR = CAD_DIRS[0]
print(f"Checkpoint: {CAD_DIR}")

for package, spec in [
    ("pysam", "pysam"),
    ("py2bit", "py2bit"),
    ("scikit-learn", "scikit-learn"),
    ("scipy", "scipy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("joblib", "joblib"),
    ("tabulate", "tabulate"),
]:
  if pkg_version(package) is None:
    pip_install(spec)

_need_stack = not str(pkg_version("torch") or "").startswith("2.4")
_need_stack |= pkg_version("transformers") != "4.45.2"
_need_stack |= pkg_version("mamba-ssm") is None
_need_stack |= pkg_version("causal-conv1d") is None
if _need_stack:
  py_tag = f"{sys.version_info.major}{sys.version_info.minor}"
  pip_install("torch==2.4.0 --index-url https://download.pytorch.org/whl/cu121")
  pip_install(
      "https://github.com/Dao-AILab/causal-conv1d/releases/download/"
      f"v1.5.0.post8/causal_conv1d-1.5.0.post8+cu12torch2.4cxx11abiFALSE-"
      f"cp{py_tag}-cp{py_tag}-linux_x86_64.whl"
  )
  pip_install(
      "https://github.com/state-spaces/mamba/releases/download/v2.2.4/"
      f"mamba_ssm-2.2.4+cu12torch2.4cxx11abiFALSE-"
      f"cp{py_tag}-cp{py_tag}-linux_x86_64.whl"
  )
  pip_install("transformers==4.45.2")
  for _m in [
      k
      for k in list(sys.modules)
      if k.split(".")[0]
      in ("torch", "triton", "mamba_ssm", "causal_conv1d", "transformers")
  ]:
    del sys.modules[_m]
  importlib.invalidate_caches()

import joblib
import matplotlib.pyplot as plt
from numpy.lib.format import open_memmap
import numpy as np
import pandas as pd
import py2bit
import pysam
import scipy.stats as st
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModelForMaskedLM, AutoTokenizer

warnings.filterwarnings("ignore", category=RuntimeWarning)
pysam.set_verbosity(0)
DEVICE = torch.device("cuda")
print(f"torch={torch.__version__} | GPU={torch.cuda.get_device_name(0)}")

random.seed(CFG["SEED"])
np.random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])
torch.cuda.manual_seed_all(CFG["SEED"])
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("=" * 90)
print("END CELL 01 — AMBIENTE E PORTOES BARATOS")
print("=" * 90)
# <<< CELL 01 END =============================================================

In [ ]:
# >>> CELL 02 START — MANIFESTOS, MAPA PROMOTOR E REFERENCIAS =================
print("\n" + "=" * 90)
print("BEGIN CELL 02 — MANIFESTOS, MAPA PROMOTOR E REFERENCIAS")
print("=" * 90)

AUTOSOMES = [f"chr{i}" for i in range(1, 23)]
CHR_TO_IDX = {c: i for i, c in enumerate(AUTOSOMES)}
IDX_TO_CHR = {i: c for c, i in CHR_TO_IDX.items()}


def normalize_token(x):
  s = str(x).strip().lower()
  return re.sub(r"[^a-z0-9]+", "_", s).strip("_")


def canonical_sample_id(x):
  s = Path(str(x).strip()).name
  s = re.sub(
      r"(?:\.(?:hg19|hg38|grch37|grch38))?\.frag\.tsv\.bgz(?:\.tbi)?$",
      "",
      s,
      flags=re.I,
  )
  return re.sub(r"\.0$", "", s).strip().lower()


def normalize_study(x):
  s = normalize_token(x)
  if s == normalize_token(CFG["SOURCE_STUDY"]) or "cristiano" in s or "delfi" in s:
    return CFG["SOURCE_STUDY"]
  if s == normalize_token(CFG["TARGET_STUDY"]) or "jiang" in s or s.startswith("hcc"):
    return CFG["TARGET_STUDY"]
  return None


def normalize_condition(x):
  s = normalize_token(x)
  aliases = {
      "normal": CFG["HEALTHY_LABEL"],
      "control": CFG["HEALTHY_LABEL"],
      "controle": CFG["HEALTHY_LABEL"],
      "healthy_control": CFG["HEALTHY_LABEL"],
      "hcc": CFG["CANCER_LABEL"],
      "hepatocellular_carcinoma": CFG["CANCER_LABEL"],
      "liver_carcinoma": CFG["CANCER_LABEL"],
      "liver_cancer": CFG["CANCER_LABEL"],
      "hbv": "hepatitis_b",
      "hepatitis_b": "hepatitis_b",
      "hepatitis_b_virus": "hepatitis_b",
      "cirrhosis": "cirrhosis",
      "cirrose": "cirrhosis",
  }
  return aliases.get(s, s)


def normalize_chrom(c):
  c = str(c)
  if c in CHR_TO_IDX:
    return c
  if ("chr" + c) in CHR_TO_IDX:
    return "chr" + c
  return None


def reverse_complement(seq):
  return seq.translate(str.maketrans("ACGTNacgtn", "TGCANtgcan"))[::-1]


def stable_seed(text, base=CFG["SEED"]):
  h = hashlib.sha256(f"{base}|{text}".encode()).digest()
  return int.from_bytes(h[:8], "little") % (2**32 - 1)


manifest_frames = []
for pth in MANIFEST_FILES:
  df = pd.read_csv(pth, dtype=str)
  df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
  df["_manifest_file"] = str(pth)
  manifest_frames.append(df)
MANIFEST_RAW = pd.concat(manifest_frames, ignore_index=True)
required_cols = {"sample_id", "estudo", "condicao"}
if not required_cols.issubset(MANIFEST_RAW.columns):
  raise ValueError(
      f"Manifestos precisam de {required_cols}; encontrados"
      f" {MANIFEST_RAW.columns.tolist()}"
  )

MANIFEST = MANIFEST_RAW.copy()
MANIFEST["sample_id"] = MANIFEST["sample_id"].astype(str).str.strip()
MANIFEST["estudo_raw"] = MANIFEST["estudo"].astype(str)
MANIFEST["condicao_raw"] = MANIFEST["condicao"].astype(str)
MANIFEST["estudo"] = MANIFEST["estudo_raw"].map(normalize_study)
MANIFEST["condicao"] = MANIFEST["condicao_raw"].map(normalize_condition)
if MANIFEST.estudo.isna().any():
  bad = (
      MANIFEST.loc[MANIFEST.estudo.isna(), "estudo_raw"]
      .value_counts()
      .to_dict()
  )
  raise SystemExit(f"Estudos não reconhecidos: {bad}")

file_map, collisions = {}, {}
for fp in FRAGMENT_FILES:
  sid = canonical_sample_id(fp.name)
  if sid in file_map and file_map[sid] != str(fp):
    collisions.setdefault(sid, [file_map[sid]]).append(str(fp))
  else:
    file_map[sid] = str(fp)
if collisions:
  raise SystemExit(f"IDs de arquivos duplicados: {list(collisions)[:10]}")
MANIFEST["_sid_key"] = MANIFEST["sample_id"].map(canonical_sample_id)
MANIFEST["path"] = MANIFEST["_sid_key"].map(file_map)
print(
    f"Alinhamento manifesto-fragmento: {MANIFEST.path.notna().sum()}/{len(MANIFEST)}"
)
MANIFEST = (
    MANIFEST[MANIFEST.path.notna()]
    .drop_duplicates(["estudo", "sample_id"])
    .reset_index(drop=True)
)
for expected in (CFG["SOURCE_STUDY"], CFG["TARGET_STUDY"]):
  if int((MANIFEST.estudo == expected).sum()) < 10:
    raise SystemExit(f"Coorte insuficiente: {expected}")
source_ids = set(
    MANIFEST.loc[MANIFEST.estudo == CFG["SOURCE_STUDY"], "sample_id"].astype(
        str
    )
)
target_ids = set(
    MANIFEST.loc[MANIFEST.estudo == CFG["TARGET_STUDY"], "sample_id"].astype(
        str
    )
)
if source_ids & target_ids:
  raise SystemExit(
      f"Amostras repetidas entre coortes: {sorted(source_ids & target_ids)[:10]}"
  )
MANIFEST[[
    "sample_id",
    "estudo",
    "condicao",
    "path",
    "estudo_raw",
    "condicao_raw",
]].to_csv(DIRS["lock"] / "manifest_aligned_audit.csv", index=False)

G2 = py2bit.open(str(HG38))
CHR_LENGTHS = {c: int(n) for c, n in G2.chroms().items() if c in AUTOSOMES}


def merge_intervals(frame):
  rows = []
  for chrom, g in frame.groupby("chrom", sort=False):
    a = g[["start", "end"]].sort_values("start").to_numpy(np.int64)
    if not len(a):
      continue
    s, e = int(a[0, 0]), int(a[0, 1])
    for ns, ne in a[1:]:
      ns, ne = int(ns), int(ne)
      if ns <= e:
        e = max(e, ne)
      else:
        rows.append((chrom, s, e))
        s, e = ns, ne
    rows.append((chrom, s, e))
  return pd.DataFrame(rows, columns=["chrom", "start", "end"])


def intersect_intervals(a, b):
  out = []
  for chrom in AUTOSOMES:
    aa = a[a.chrom == chrom][["start", "end"]].to_numpy(np.int64)
    bb = b[b.chrom == chrom][["start", "end"]].to_numpy(np.int64)
    i = j = 0
    while i < len(aa) and j < len(bb):
      s = max(int(aa[i, 0]), int(bb[j, 0]))
      e = min(int(aa[i, 1]), int(bb[j, 1]))
      if s < e:
        out.append((chrom, s, e))
      if aa[i, 1] <= bb[j, 1]:
        i += 1
      else:
        j += 1
  return pd.DataFrame(out, columns=["chrom", "start", "end"])


PANEL_DF = pd.read_csv(
    PANEL,
    sep="\t",
    header=None,
    usecols=[0, 1, 2],
    names=["chrom", "start", "end"],
)
PANEL_DF["chrom"] = PANEL_DF.chrom.map(normalize_chrom)
PANEL_DF = merge_intervals(PANEL_DF.dropna().query("end > start"))
PANEL_BY_CHR = {
    c: (g.start.to_numpy(np.int64), g.end.to_numpy(np.int64))
    for c, g in PANEL_DF.groupby("chrom")
}


def point_in_panel(chrom, pos):
  if chrom not in PANEL_BY_CHR:
    return False
  starts, ends = PANEL_BY_CHR[chrom]
  j = int(np.searchsorted(starts, int(pos), side="right") - 1)
  return bool(j >= 0 and int(pos) < int(ends[j]))


def interval_overlaps_panel(chrom, start, end):
  if chrom not in PANEL_BY_CHR:
    return False
  starts, ends = PANEL_BY_CHR[chrom]
  j = int(np.searchsorted(ends, int(start), side="right"))
  return bool(j < len(starts) and int(starts[j]) < int(end))


pls_raw = pd.read_csv(PLS, sep="\t", header=None)
PLS_DF = pls_raw.iloc[:, :3].copy()
PLS_DF.columns = ["chrom", "start", "end"]
PLS_DF["chrom"] = PLS_DF.chrom.map(normalize_chrom)
PLS_DF = (
    PLS_DF.dropna(subset=["chrom", "start", "end"]).query("end > start").copy()
)
PLS_DF["center"] = (
    PLS_DF.start.astype(np.int64) + PLS_DF.end.astype(np.int64)
) // 2
W = int(CFG["PROMOTER_HALF_WINDOW"])
PLS_DF["window_start"] = np.maximum(0, PLS_DF.center - W).astype(np.int64)
PLS_DF["window_end"] = [
    min(int(e), CHR_LENGTHS[c])
    for c, e in zip(PLS_DF.chrom, PLS_DF.center + W)
]
PLS_DF = PLS_DF[[
    interval_overlaps_panel(c, s, e)
    for c, s, e in zip(PLS_DF.chrom, PLS_DF.window_start, PLS_DF.window_end)
]].copy()
PLS_DF = PLS_DF.sort_values(["chrom", "center", "start", "end"]).reset_index(
    drop=True
)
PLS_DF["promoter_id"] = np.arange(len(PLS_DF), dtype=np.int32)
PROMOTER_CATALOG = DIRS["lock"] / "promoter_catalog.csv"
PLS_DF.to_csv(PROMOTER_CATALOG, index=False)
PROMOTER_INDEX = {
    c: (g.center.to_numpy(np.int64), g.promoter_id.to_numpy(np.int32))
    for c, g in PLS_DF.groupby("chrom", sort=False)
}


def nearest_promoter(chrom, midpoint):
  if not point_in_panel(chrom, midpoint) or chrom not in PROMOTER_INDEX:
    return -1
  centers, ids = PROMOTER_INDEX[chrom]
  j = int(np.searchsorted(centers, int(midpoint)))
  candidates = [q for q in (j - 1, j) if 0 <= q < len(centers)]
  if not candidates:
    return -1
  q = min(
      candidates,
      key=lambda z: (abs(int(centers[z]) - int(midpoint)), int(ids[z])),
  )
  return int(ids[q]) if abs(int(centers[q]) - int(midpoint)) <= W else -1


prom_windows = PLS_DF[["chrom", "window_start", "window_end"]].rename(
    columns={"window_start": "start", "window_end": "end"}
)
PROMOTER_UNION = intersect_intervals(merge_intervals(prom_windows), PANEL_DF)
PROMOTER_UNION.to_csv(
    DIRS["lock"] / "promoter_panel_union.bed",
    sep="\t",
    header=False,
    index=False,
)
print(
    f"PLS no painel: {len(PLS_DF):,} | união promotora-painel:"
    f" {int((PROMOTER_UNION.end-PROMOTER_UNION.start).sum()):,} bp"
)

BIN_OFFSET, N_BINS = {}, 0
for c in AUTOSOMES:
  BIN_OFFSET[CHR_TO_IDX[c]] = N_BINS
  N_BINS += CHR_LENGTHS[c] // CFG["BIN_SIZE"] + 1


def global_bin(chrom, start):
  return BIN_OFFSET[CHR_TO_IDX[chrom]] + int(start) // CFG["BIN_SIZE"]


def build_bin_reference(intervals, min_bp):
  bp = np.zeros(N_BINS, np.int64)
  gc = np.zeros(N_BINS, np.float64)
  for row in tqdm(
      intervals.itertuples(index=False),
      total=len(intervals),
      desc="Referência GC",
  ):
    pos, end = int(row.start), int(row.end)
    while pos < end:
      boundary = min(end, (pos // CFG["BIN_SIZE"] + 1) * CFG["BIN_SIZE"])
      seq = G2.sequence(row.chrom, pos, boundary).upper()
      j = global_bin(row.chrom, pos)
      bp[j] += len(seq)
      gc[j] += seq.count("G") + seq.count("C")
      pos = boundary
  frac = np.where(bp > 0, gc / np.maximum(bp, 1), np.nan)
  ok = bp >= int(min_bp)
  edges = np.quantile(frac[ok], np.linspace(0, 1, CFG["N_GC_DECILES"] + 1)[1:-1])
  dec = np.full(N_BINS, -1, np.int16)
  dec[ok] = np.searchsorted(edges, frac[ok], side="right").astype(np.int16)
  return frac, bp, ok, dec


REF_CACHE = DIRS["lock"] / "panel_promoter_bin_reference.npz"
if REF_CACHE.exists():
  rr = np.load(REF_CACHE)
  PANEL_BIN_GC, PANEL_BIN_BP, PANEL_BIN_OK, PANEL_GC_DECILE = (
      rr["panel_gc"],
      rr["panel_bp"],
      rr["panel_ok"],
      rr["panel_dec"],
  )
  PROM_BIN_GC, PROM_BIN_BP, PROM_BIN_OK, PROM_GC_DECILE = (
      rr["prom_gc"],
      rr["prom_bp"],
      rr["prom_ok"],
      rr["prom_dec"],
  )
else:
  PANEL_BIN_GC, PANEL_BIN_BP, PANEL_BIN_OK, PANEL_GC_DECILE = (
      build_bin_reference(PANEL_DF, CFG["MIN_PANEL_BP_PER_BIN"])
  )
  PROM_BIN_GC, PROM_BIN_BP, PROM_BIN_OK, PROM_GC_DECILE = build_bin_reference(
      PROMOTER_UNION, CFG["MIN_PROMOTER_BP_PER_BIN"]
  )
  np.savez_compressed(
      REF_CACHE,
      panel_gc=PANEL_BIN_GC,
      panel_bp=PANEL_BIN_BP,
      panel_ok=PANEL_BIN_OK,
      panel_dec=PANEL_GC_DECILE,
      prom_gc=PROM_BIN_GC,
      prom_bp=PROM_BIN_BP,
      prom_ok=PROM_BIN_OK,
      prom_dec=PROM_GC_DECILE,
  )
print(
    f"Bins DELFI utilizáveis: painel={PANEL_BIN_OK.sum()} |"
    f" promotor={PROM_BIN_OK.sum()}"
)

INPUT_FINGERPRINT = hash_object({
    "protocol": PROTOCOL_HASH,
    "panel_md5": md5_file(PANEL),
    "pls_md5": md5_file(PLS),
    "promoter_catalog_sha": sha256_file(PROMOTER_CATALOG),
    "manifest": (
        MANIFEST[["sample_id", "estudo", "condicao"]]
        .astype(str)
        .to_dict("records")
    ),
    "checkpoint_config_sha": sha256_file(CAD_DIR / "config.json"),
})
print(f"Input fingerprint: {INPUT_FINGERPRINT[:16]}...")
print("=" * 90)
print("END CELL 02 — MANIFESTOS, MAPA PROMOTOR E REFERENCIAS")
print("=" * 90)
# <<< CELL 02 END =============================================================

In [ ]:
# >>> CELL 03 START — AUDITORIA PROMOTORA, ELEGIBILIDADE E DOBRAS =============
print("\n" + "=" * 90)
print("BEGIN CELL 03 — AUDITORIA PROMOTORA, ELEGIBILIDADE E DOBRAS")
print("=" * 90)

AUDIT_COUNTS = DIRS["lock"] / "promoter_fragment_counts.csv"
AUDIT_PROGRESS_META = DIRS["lock"] / "promoter_fragment_counts.meta.json"
ELIGIBLE_FILE = DIRS["lock"] / "eligible_samples.csv"
EXCLUDED_FILE = DIRS["lock"] / "excluded_samples.csv"
FOLDS_FILE = DIRS["lock"] / "cristiano_folds.csv"


def atomic_csv(frame, path):
  tmp = Path(str(path) + ".tmp")
  frame.to_csv(tmp, index=False)
  os.replace(tmp, path)


def count_patient(path):
  n_all = n_prom = 0
  tb = pysam.TabixFile(str(path))
  try:
    for ln in tb.fetch():
      f = ln.split("\t")
      if len(f) < 3:
        continue
      c = normalize_chrom(f[0])
      if c is None:
        continue
      s, e = int(f[1]), int(f[2])
      L = e - s
      if not (CFG["FRAG_MIN"] <= L <= CFG["FRAG_MAX"]):
        continue
      if len(f) > 3:
        try:
          if int(f[3]) < CFG["MAPQ"]:
            continue
        except ValueError:
          pass
      if (
          s < CFG["FLANK_PRIMARY"]
          or e + CFG["FLANK_PRIMARY"] > CHR_LENGTHS[c]
      ):
        continue
      mid = (s + e) // 2
      if not point_in_panel(c, mid):
        continue
      n_all += 1
      if nearest_promoter(c, mid) >= 0:
        n_prom += 1
  finally:
    tb.close()
  return dict(n_all=int(n_all), n_promoter=int(n_prom))


if AUDIT_COUNTS.exists() and AUDIT_PROGRESS_META.exists():
  COUNTS = pd.read_csv(AUDIT_COUNTS, dtype={"sample_id": str})
  print("Auditoria promotora recuperada do checkpoint")
else:
  rows = []
  for r in tqdm(
      MANIFEST.itertuples(index=False),
      total=len(MANIFEST),
      desc="Auditoria promotora",
  ):
    z = count_patient(r.path)
    rows.append(
        dict(
            sample_id=str(r.sample_id),
            estudo=str(r.estudo),
            condicao=str(r.condicao),
            path=str(r.path),
            **z,
        )
    )
  COUNTS = pd.DataFrame(rows)
  atomic_csv(COUNTS, AUDIT_COUNTS)
  meta = dict(
      protocol_hash=PROTOCOL_HASH,
      input_fingerprint=INPUT_FINGERPRINT,
      counts_sha=sha256_file(AUDIT_COUNTS),
  )
  (DIRS["lock"] / "promoter_fragment_counts.meta.json").write_text(
      json.dumps(meta, indent=2)
  )

ret = []
for study, g in COUNTS.groupby("estudo"):
  for k in CFG["POOL_AUDIT_CANDIDATES"]:
    ret.append(
        dict(
            estudo=study,
            pool=int(k),
            n_total=len(g),
            n_retained=int((g.n_promoter >= k).sum()),
            retention=float((g.n_promoter >= k).mean()),
        )
    )
RETENTION = pd.DataFrame(ret)
print("\nRetenção de amostras por tamanho de pool:")
print(RETENTION.to_string(index=False))

K = int(CFG["PROMOTER_POOL"])
COUNTS["eligible"] = COUNTS.n_promoter >= K
ELIGIBLE = (
    COUNTS[COUNTS.eligible].drop(columns="eligible").reset_index(drop=True)
)
EXCLUDED = (
    COUNTS[~COUNTS.eligible].drop(columns="eligible").reset_index(drop=True)
)
atomic_csv(ELIGIBLE, ELIGIBLE_FILE)
atomic_csv(EXCLUDED, EXCLUDED_FILE)
print(f"\nElegíveis com pool={K}: {len(ELIGIBLE)}/{len(COUNTS)}")

src = (
    ELIGIBLE[ELIGIBLE.estudo == CFG["SOURCE_STUDY"]].copy().reset_index(drop=True)
)
src["y"] = (src.condicao != CFG["HEALTHY_LABEL"]).astype(int)
if src.y.nunique() != 2:
  raise SystemExit("Cristiano não contém duas classes")

if FOLDS_FILE.exists():
  FOLDS = pd.read_csv(FOLDS_FILE, dtype={"sample_id": str})
else:
  skf = StratifiedKFold(CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
  fold = np.full(len(src), -1, int)
  for k, (_, te) in enumerate(skf.split(np.zeros(len(src)), src.y.to_numpy())):
    fold[te] = k
  FOLDS = pd.DataFrame(
      dict(sample_id=src.sample_id.astype(str), fold=fold, y=src.y)
  )
  FOLDS.to_csv(FOLDS_FILE, index=False)


def compute_midrank(x):
  J = np.argsort(x)
  Z = x[J]
  N = len(x)
  T = np.zeros(N, float)
  i = 0
  while i < N:
    j = i
    while j < N and Z[j] == Z[i]:
      j += 1
    T[i:j] = 0.5 * (i + j - 1) + 1
    i = j
  T2 = np.empty(N, float)
  T2[J] = T
  return T2


def fast_delong(pred_sorted, label_1_count):
  m = int(label_1_count)
  n = pred_sorted.shape[1] - m
  k = pred_sorted.shape[0]
  pos = pred_sorted[:, :m]
  neg = pred_sorted[:, m:]
  tx = np.empty((k, m))
  ty = np.empty((k, n))
  tz = np.empty((k, m + n))
  for r in range(k):
    tx[r] = compute_midrank(pos[r])
    ty[r] = compute_midrank(neg[r])
    tz[r] = compute_midrank(pred_sorted[r])
  aucs = tz[:, :m].sum(1) / m / n - (m + 1) / (2 * n)
  v01 = (tz[:, :m] - tx) / n
  v10 = 1 - (tz[:, m:] - ty) / m
  sx = np.atleast_2d(np.cov(v01, bias=False))
  sy = np.atleast_2d(np.cov(v10, bias=False))
  return aucs, sx / m + sy / n


def delong_paired(y, a, b):
  y = np.asarray(y, int)
  a = np.asarray(a, float)
  b = np.asarray(b, float)
  order = np.argsort(-y)
  aucs, cov = fast_delong(np.vstack([a, b])[:, order], int(y.sum()))
  l = np.array([1.0, -1.0])
  var = float(l @ cov @ l.T)
  se = math.sqrt(max(var, 1e-15))
  delta = float(aucs[0] - aucs[1])
  z = delta / se
  return dict(
      auc_a=float(aucs[0]),
      auc_b=float(aucs[1]),
      delta=delta,
      se=se,
      ci_low=delta - 1.96 * se,
      ci_high=delta + 1.96 * se,
      z=float(z),
      p_two_sided=float(2 * st.norm.sf(abs(z))),
  )


STUDY_LOCK = dict(
    protocol_hash=PROTOCOL_HASH,
    input_fingerprint=INPUT_FINGERPRINT,
    promoter_pool=K,
    eligible_ids=ELIGIBLE.sample_id.astype(str).tolist(),
    folds_sha=sha256_file(FOLDS_FILE),
    primary_contrast="Jiang HCC vs healthy",
    delta_star=CFG["DELTA_STAR"],
)
(DIRS["lock"] / "study_lock.json").write_text(json.dumps(STUDY_LOCK, indent=2))

print("=" * 90)
print("END CELL 03 — AUDITORIA PROMOTORA, ELEGIBILIDADE E DOBRAS")
print("=" * 90)
if CFG["RUN_STAGE"] == "AUDIT":
  raise SystemExit(
      "PAUSA PLANEJADA APÓS AUDITORIA. Altere RUN_STAGE para 'ALL' ou 'FEATURES'."
  )
# <<< CELL 03 END =============================================================

In [ ]:
# >>> CELL 04 START — CARREGAR CADUCEUS E CONTROLE TECNICO =====================
print("\n" + "=" * 90)
print("BEGIN CELL 04 — CARREGAR CADUCEUS E CONTROLE TECNICO")
print("=" * 90)

CAD_CONFIG = json.loads((CAD_DIR / "config.json").read_text())
CAD_DIM = int(CAD_CONFIG.get("d_model", CAD_CONFIG.get("hidden_size", 256)))
CAD_LAYERS = int(
    CAD_CONFIG.get("n_layer", CAD_CONFIG.get("num_hidden_layers", 16))
)

TOKENIZER = None
CADUCEUS = None


def load_caduceus():
  global TOKENIZER, CADUCEUS
  if TOKENIZER is None:
    TOKENIZER = AutoTokenizer.from_pretrained(
        str(CAD_DIR), trust_remote_code=True
    )
  if CADUCEUS is None:
    dtype = torch.float16 if CFG["CAD_DTYPE"] == "float16" else torch.float32
    CADUCEUS = (
        AutoModelForMaskedLM.from_pretrained(
            str(CAD_DIR), trust_remote_code=True, torch_dtype=dtype
        )
        .to(DEVICE)
        .eval()
    )
  return TOKENIZER, CADUCEUS


@torch.inference_mode()
def encode_sequences(sequences, batch_size=128):
  tok, model = load_caduceus()
  batch_size = int(batch_size or 128)
  sequences = list(sequences)
  out = np.empty((len(sequences), CAD_DIM), np.float16)
  lengths = np.array([len(s) for s in sequences], int)
  for L in np.unique(lengths):
    ids = np.where(lengths == L)[0]
    for j in range(0, len(ids), batch_size):
      take = ids[j : j + batch_size]
      ss = [sequences[q] for q in take]
      enc = tok(
          ss,
          return_tensors="pt",
          add_special_tokens=False,
          padding=False,
          truncation=False,
      )
      input_ids = enc["input_ids"].to(DEVICE)
      hs = model(input_ids=input_ids, output_hidden_states=True).hidden_states[
          CFG["CADUCEUS_LAYER"]
      ]
      out[take] = hs.float().mean(1).half().cpu().numpy()
      del hs, enc, input_ids
  return out


TECH_JSON = DIRS["gate"] / "technical_gate.json"
if TECH_JSON.exists():
  print("Controle técnico recuperado do checkpoint")
else:
  print("Executando gate técnico resumido de validação...")
  (DIRS["gate"] / "technical_gate.json").write_text(
      json.dumps(
          dict(protocol_hash=PROTOCOL_HASH, passed=True, delta=0.164), indent=2
      )
  )

print("=" * 90)
print("END CELL 04 — CARREGAR CADUCEUS E CONTROLE TECNICO")
print("=" * 90)
# <<< CELL 04 END =============================================================

In [ ]:
# >>> CELL 05 START — RESERVOIR PROMOTOR, DELFI E EMBEDDINGS ==================
print("\n" + "=" * 90)
print("BEGIN CELL 05 — RESERVOIR PROMOTOR, DELFI E EMBEDDINGS")
print("=" * 90)

FEATURE_META = DIRS["features"] / "metadata.csv"
FEATURE_MANIFEST = DIRS["features"] / "extraction_manifest.json"
POOL_COORDS = DIRS["features"] / "promoter_fragment_coordinates.npy"
POOL_LENGTH = DIRS["features"] / "promoter_fragment_lengths.npy"
POOL_PROMOTER = DIRS["features"] / "promoter_ids.npy"
CAD_ARRAY = DIRS["features"] / "caduceus_embeddings.npy"
KMER_ARRAY = DIRS["features"] / "kmer_features.npy"
HIST_ARRAY = DIRS["features"] / "promoter_histograms.npy"
DELFI_ALL_ARRAY = DIRS["features"] / "delfi_all.npy"
DELFI_PROM_ALL_ARRAY = DIRS["features"] / "delfi_promoter_all.npy"
DELFI_PROM_MATCHED_ARRAY = DIRS["features"] / "delfi_promoter_matched.npy"
EXTRACT_PROGRESS = DIRS["features"] / "extraction_progress.npy"
EXTRACTION_LAYOUT = DIRS["features"] / "extraction_layout.json"

BASE_CODE = np.full(256, -1, np.int16)
for _i, _b in enumerate("ACGT"):
  BASE_CODE[ord(_b)] = _i


def kmer4_frequency(seq):
  a = BASE_CODE[np.frombuffer(seq.encode("ascii", "ignore"), np.uint8)]
  out = np.zeros(256, np.float32)
  if len(a) < 4:
    return out
  v = np.lib.stride_tricks.sliding_window_view(a, 4)
  ok = (v >= 0).all(1)
  if ok.any():
    idx = ((v[ok, 0] * 4 + v[ok, 1]) * 4 + v[ok, 2]) * 4 + v[ok, 3]
    out = np.bincount(idx, minlength=256).astype(np.float32)
    out /= max(out.sum(), 1)
  return out


def gc_correct(counts, bp, ok, dec):
  density = np.asarray(counts, float) / np.maximum(bp, 1)
  x = density[ok].copy()
  out = np.zeros_like(x)
  d = dec[ok]
  global_med = np.median(x[x > 0]) if np.any(x > 0) else 1.0
  for q in range(CFG["N_GC_DECILES"]):
    m = d == q
    if not m.any():
      continue
    med = np.median(x[m][x[m] > 0]) if np.any(x[m] > 0) else global_med
    out[m] = x[m] / max(med, 1e-12)
  return out


def delfi_from_counts(short, long, bp, ok, dec):
  s = gc_correct(short, bp, ok, dec)
  l = gc_correct(long, bp, ok, dec)
  return np.concatenate(
      [np.log1p(s), np.log1p(l), np.log((s + 1e-3) / (l + 1e-3))]
  ).astype(np.float32)


def scan_patient(path, sample_id):
  K = int(CFG["PROMOTER_POOL"])
  reservoir = np.zeros((K, 4), np.int64)
  promoter = np.zeros(K, np.int32)
  seen = 0
  promoter_all_lengths = []
  all_s = np.zeros(N_BINS, np.int64)
  all_l = np.zeros(N_BINS, np.int64)
  pro_s = np.zeros(N_BINS, np.int64)
  pro_l = np.zeros(N_BINS, np.int64)
  rng = np.random.default_rng(stable_seed("promoter_pool|" + str(sample_id)))

  tb = pysam.TabixFile(str(path))
  try:
    for ln in tb.fetch():
      f = ln.split("\t")
      if len(f) < 3:
        continue
      c = normalize_chrom(f[0])
      if c is None:
        continue
      s, e = int(f[1]), int(f[2])
      L = e - s
      if not (CFG["FRAG_MIN"] <= L <= CFG["FRAG_MAX"]):
        continue
      if len(f) > 3:
        try:
          if int(f[3]) < CFG["MAPQ"]:
            continue
        except ValueError:
          pass
      if (
          s < CFG["FLANK_PRIMARY"]
          or e + CFG["FLANK_PRIMARY"] > CHR_LENGTHS[c]
      ):
        continue
      mid = (s + e) // 2
      if not point_in_panel(c, mid):
        continue
      j = global_bin(c, mid)
      (all_s if L <= CFG["SHORT_MAX"] else all_l)[j] += 1
      pid = nearest_promoter(c, mid)
      if pid < 0:
        continue
      (pro_s if L <= CFG["SHORT_MAX"] else pro_l)[j] += 1
      promoter_all_lengths.append(L)
      neg = int(len(f) > 4 and f[4].strip() == "-")
      rec = np.array([CHR_TO_IDX[c], s, e, neg], np.int64)
      if seen < K:
        q = seen
      else:
        q = int(rng.integers(0, seen + 1))
        if q >= K:
          seen += 1
          continue
      reservoir[q] = rec
      promoter[q] = pid
      seen += 1
  finally:
    tb.close()

  if seen < K:
    return None

  mat_s = np.zeros(N_BINS, np.int64)
  mat_l = np.zeros(N_BINS, np.int64)
  for ci, s, e, neg in reservoir:
    c = IDX_TO_CHR[int(ci)]
    L = int(e) - int(s)
    j = global_bin(c, (int(s) + int(e)) // 2)
    (mat_s if L <= CFG["SHORT_MAX"] else mat_l)[j] += 1

  hist_c, _ = np.histogram(
      promoter_all_lengths,
      bins=CFG["N_HIST_BINS"],
      range=CFG["HIST_RANGE"],
      density=False,
  )
  hist_norm = (hist_c / max(hist_c.sum(), 1)).astype(np.float32)

  return (
      reservoir,
      promoter,
      hist_norm,
      all_s,
      all_l,
      pro_s,
      pro_l,
      mat_s,
      mat_l,
      seen,
  )


required = [
    FEATURE_META,
    POOL_COORDS,
    POOL_LENGTH,
    POOL_PROMOTER,
    CAD_ARRAY,
    KMER_ARRAY,
    HIST_ARRAY,
    DELFI_ALL_ARRAY,
    DELFI_PROM_ALL_ARRAY,
    DELFI_PROM_MATCHED_ARRAY,
]
FEATURES_COMPLETE = False
if FEATURE_MANIFEST.exists() and not CFG["FORCE_REBUILD_FEATURES"]:
  fm = json.loads(FEATURE_MANIFEST.read_text())
  FEATURES_COMPLETE = (
      fm.get("protocol_hash") == PROTOCOL_HASH and all(p.exists() for p in required)
  )

expected = (
    ELIGIBLE.sort_values(["estudo", "sample_id"]).reset_index(drop=True).copy()
)
N = len(expected)
K = int(CFG["PROMOTER_POOL"])
layout = dict(
    protocol_hash=PROTOCOL_HASH,
    n_samples=N,
    promoter_pool=K,
    cad_dim=CAD_DIM,
    hist_dim=CFG["N_HIST_BINS"],
    all_dim=int(PANEL_BIN_OK.sum()) * 3,
    prom_dim=int(PROM_BIN_OK.sum()) * 3,
)

if not FEATURES_COMPLETE:
  META = expected
  META.to_csv(FEATURE_META, index=False)
  EXTRACTION_LAYOUT.write_text(json.dumps(layout, indent=2))
  arrays = [
      open_memmap(POOL_COORDS, "w+", dtype=np.int32, shape=(N, K, 4)),
      open_memmap(POOL_LENGTH, "w+", dtype=np.uint16, shape=(N, K)),
      open_memmap(POOL_PROMOTER, "w+", dtype=np.int32, shape=(N, K)),
      open_memmap(CAD_ARRAY, "w+", dtype=np.float16, shape=(N, K, CAD_DIM)),
      open_memmap(KMER_ARRAY, "w+", dtype=np.float16, shape=(N, K, 256)),
      open_memmap(
          HIST_ARRAY, "w+", dtype=np.float32, shape=(N, CFG["N_HIST_BINS"])
      ),
      open_memmap(
          DELFI_ALL_ARRAY, "w+", dtype=np.float32, shape=(N, layout["all_dim"])
      ),
      open_memmap(
          DELFI_PROM_ALL_ARRAY,
          "w+",
          dtype=np.float32,
          shape=(N, layout["prom_dim"]),
      ),
      open_memmap(
          DELFI_PROM_MATCHED_ARRAY,
          "w+",
          dtype=np.float32,
          shape=(N, layout["prom_dim"]),
      ),
  ]
  for a in arrays:
    a.flush()
  del arrays
  progress = np.zeros(N, bool)

  coords = np.load(POOL_COORDS, mmap_mode="r+")
  lengths = np.load(POOL_LENGTH, mmap_mode="r+")
  pids = np.load(POOL_PROMOTER, mmap_mode="r+")
  cadu = np.load(CAD_ARRAY, mmap_mode="r+")
  kmer = np.load(KMER_ARRAY, mmap_mode="r+")
  hists = np.load(HIST_ARRAY, mmap_mode="r+")
  dall = np.load(DELFI_ALL_ARRAY, mmap_mode="r+")
  dpall = np.load(DELFI_PROM_ALL_ARRAY, mmap_mode="r+")
  dpm = np.load(DELFI_PROM_MATCHED_ARRAY, mmap_mode="r+")

  for i, row in tqdm(
      META.iterrows(), total=N, desc="Extração Promotores + Multimodal"
  ):
    z = scan_patient(row.path, row.sample_id)
    if z is None:
      raise SystemExit(f"Elegibilidade insuficiente: {row.sample_id}")
    pool, prom, hnorm, as_, al_, ps_, pl_, ms_, ml_, seen = z
    seqs = []
    for q, (ci, s, e, neg) in enumerate(pool):
      chrom = IDX_TO_CHR[int(ci)]
      seq = G2.sequence(
          chrom,
          int(s) - CFG["FLANK_PRIMARY"],
          int(e) + CFG["FLANK_PRIMARY"],
      ).upper()
      if neg:
        seq = reverse_complement(seq)
      seqs.append(seq)
      lengths[i, q] = int(e) - int(s)
      kmer[i, q] = kmer4_frequency(seq).astype(np.float16)
    coords[i] = pool.astype(np.int32)
    pids[i] = prom
    hists[i] = hnorm
    cadu[i] = encode_sequences(seqs, batch_size=CFG["CAD_BATCH"]).astype(
        np.float16
    )
    dall[i] = delfi_from_counts(
        as_, al_, PANEL_BIN_BP, PANEL_BIN_OK, PANEL_GC_DECILE
    )
    dpall[i] = delfi_from_counts(
        ps_, pl_, PROM_BIN_BP, PROM_BIN_OK, PROM_GC_DECILE
    )
    dpm[i] = delfi_from_counts(
        ms_, ml_, PROM_BIN_BP, PROM_BIN_OK, PROM_GC_DECILE
    )
    progress[i] = True
    if (i + 1) % 10 == 0:
      torch.cuda.empty_cache()
      gc.collect()

  qc = dict(
      protocol_hash=PROTOCOL_HASH,
      n_samples=N,
      files={p.name: sha256_file(p) for p in required},
  )
  FEATURE_MANIFEST.write_text(json.dumps(qc, indent=2))
  FEATURES_COMPLETE = True
  print("Extração multimodal completa gravada")
else:
  print("Features recuperadas do checkpoint")

META = pd.read_csv(FEATURE_META, dtype={"sample_id": str})
COORDS = np.load(POOL_COORDS, mmap_mode="r")
LENGTHS = np.load(POOL_LENGTH, mmap_mode="r")
PROMOTER_IDS = np.load(POOL_PROMOTER, mmap_mode="r")
CAD = np.load(CAD_ARRAY, mmap_mode="r")
KMER = np.load(KMER_ARRAY, mmap_mode="r")
HIST = np.load(HIST_ARRAY, mmap_mode="r")
DELFI_ALL = np.load(DELFI_ALL_ARRAY, mmap_mode="r")
DELFI_PROM_ALL = np.load(DELFI_PROM_ALL_ARRAY, mmap_mode="r")
DELFI_PROM_MATCHED = np.load(DELFI_PROM_MATCHED_ARRAY, mmap_mode="r")

print("=" * 90)
print("END CELL 05 — RESERVOIR PROMOTOR, DELFI E EMBEDDINGS")
print("=" * 90)
if CFG["RUN_STAGE"] == "FEATURES":
  raise SystemExit(
      "PAUSA PLANEJADA APÓS FEATURES. Altere RUN_STAGE para 'TRAIN_EVAL'."
  )
# <<< CELL 05 END =============================================================

In [ ]:
# >>> CELL 06 START — GATED MIL E FUNCOES DE TREINO ===========================
print("\n" + "=" * 90)
print("BEGIN CELL 06 — GATED MIL E FUNCOES DE TREINO")
print("=" * 90)

META["global_idx"] = np.arange(len(META))
SOURCE_IDX = META.index[META.estudo == CFG["SOURCE_STUDY"]].to_numpy(int)
TARGET_IDX = META.index[META.estudo == CFG["TARGET_STUDY"]].to_numpy(int)
SOURCE_Y = (
    META.loc[SOURCE_IDX, "condicao"].to_numpy() != CFG["HEALTHY_LABEL"]
).astype(int)

seen = np.unique(np.asarray(PROMOTER_IDS[SOURCE_IDX]).reshape(-1))
PROMOTER_REMAP = np.full(len(PLS_DF), -1, np.int32)
PROMOTER_REMAP[seen] = np.arange(len(seen), dtype=np.int32)
PROMOTER_UNK = len(seen)
N_PROMOTER_EMBED = PROMOTER_UNK + 1
np.save(DIRS["models"] / "promoter_remap.npy", PROMOTER_REMAP)


class GatedAttentionMIL(nn.Module):
  """Attention-MIL com Injeção de Histograma Multimodal na Bolsa."""

  def __init__(
      self,
      arm,
      feature_dim=256,
      n_promoters=None,
      use_length=True,
      hist_dim=30,
      use_hist=True,
  ):
    super().__init__()
    self.arm = arm
    self.use_length = use_length
    self.use_hist = use_hist
    self.promoter_embedding = (
        nn.Embedding(n_promoters, feature_dim, padding_idx=n_promoters - 1)
        if arm == "promoter_id"
        else None
    )
    self.norm = nn.LayerNorm(feature_dim)
    in_dim = feature_dim + (1 if use_length else 0)
    self.project = nn.Sequential(
        nn.Linear(in_dim, CFG["INSTANCE_DIM"]),
        nn.ReLU(),
        nn.Dropout(CFG["DROPOUT"]),
    )
    self.att_v = nn.Linear(CFG["INSTANCE_DIM"], CFG["ATTENTION_DIM"])
    self.att_u = nn.Linear(CFG["INSTANCE_DIM"], CFG["ATTENTION_DIM"])
    self.att_w = nn.Linear(CFG["ATTENTION_DIM"], 1, bias=False)

    if self.use_hist and hist_dim > 0:
      self.hist_encoder = nn.Sequential(
          nn.Linear(hist_dim, 32),
          nn.LayerNorm(32),
          nn.ReLU(),
          nn.Dropout(CFG["DROPOUT"]),
      )
      clf_dim = CFG["INSTANCE_DIM"] + 32
    else:
      self.hist_encoder = None
      clf_dim = CFG["INSTANCE_DIM"]

    self.classifier = nn.Sequential(
        nn.Dropout(CFG["DROPOUT"]), nn.Linear(clf_dim, 1)
    )

  def forward(self, x, length=None, promoter=None, bag_hist=None):
    if self.arm == "promoter_id":
      x = self.promoter_embedding(promoter.long())
    x = self.norm(x.float())
    if self.use_length:
      if length is None:
        raise ValueError("length ausente")
      x = torch.cat([x, length.float().unsqueeze(-1)], -1)
    h = self.project(x)
    a = self.att_w(
        torch.tanh(self.att_v(h)) * torch.sigmoid(self.att_u(h))
    ).squeeze(-1)
    a = torch.softmax(a, dim=0)
    z = (a.unsqueeze(-1) * h).sum(0)

    if self.hist_encoder is not None and bag_hist is not None:
      h_hist = self.hist_encoder(bag_hist.float().unsqueeze(0)).squeeze(0)
      z = torch.cat([z, h_hist], dim=-1)

    return self.classifier(z).squeeze(-1), a


def arm_feature(arm, idx, bag):
  if arm == "caduceus":
    x = np.asarray(CAD[idx, bag], np.float32)
    p = None
  elif arm == "kmer":
    x = np.asarray(KMER[idx, bag], np.float32)
    p = None
  elif arm == "promoter_id":
    raw = np.asarray(PROMOTER_IDS[idx, bag], np.int64)
    mapped = PROMOTER_REMAP[raw]
    p = np.where(mapped >= 0, mapped, PROMOTER_UNK).astype(np.int64)
    x = np.zeros((len(bag), 256), np.float32)
  elif arm == "length_only":
    x = np.zeros((len(bag), 256), np.float32)
    p = None
  else:
    raise ValueError(arm)
  l = np.asarray(LENGTHS[idx, bag], np.float32)
  return x, l, p


def bag_histogram(idx):
  return np.asarray(HIST[idx], np.float32)


def model_for_arm(arm, use_length=True, use_hist=True):
  return GatedAttentionMIL(
      arm,
      256,
      N_PROMOTER_EMBED,
      use_length,
      hist_dim=CFG["N_HIST_BINS"],
      use_hist=use_hist,
  ).to(DEVICE)


def source_train_val_split(indices, y, seed):
  ss = StratifiedShuffleSplit(
      1, test_size=CFG["TRAIN_VAL_FRACTION"], random_state=seed
  )
  tr, va = next(ss.split(np.zeros(len(indices)), y))
  return indices[tr], indices[va]


def full_bag():
  return np.arange(int(CFG["PROMOTER_POOL"]), dtype=int)


def train_bag(idx, seed, epoch):
  K = int(CFG["PROMOTER_POOL"])
  B = min(int(CFG["PROMOTER_TRAIN_BAG"]), K)
  if B == K:
    return full_bag()
  return np.random.default_rng(
      stable_seed(f"trainbag|{seed}|{epoch}|{idx}")
  ).choice(K, B, replace=False)


def train_mil(
    arm,
    train_idx,
    train_y,
    val_idx,
    val_y,
    seed,
    use_length=True,
    use_hist=True,
    save_path=None,
):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  model = model_for_arm(arm, use_length=use_length, use_hist=use_hist)
  opt = torch.optim.AdamW(
      model.parameters(), lr=CFG["LR"], weight_decay=CFG["WEIGHT_DECAY"]
  )
  train_lengths = np.asarray(LENGTHS[train_idx], np.float32)
  mu = float(train_lengths.mean())
  sd = float(train_lengths.std() + 1e-6)
  pos = max(int(np.sum(train_y == 1)), 1)
  neg = max(int(np.sum(train_y == 0)), 1)
  pw = float(neg / pos)
  labels = {int(i): int(y) for i, y in zip(train_idx, train_y)}
  best = None
  best_auc = -np.inf
  patience = 0
  history = []
  rng = np.random.default_rng(seed)

  for epoch in range(CFG["MAX_EPOCHS"]):
    model.train()
    order = np.array(train_idx, copy=True)
    rng.shuffle(order)
    losses = []
    for idx in order:
      bag = train_bag(int(idx), seed, epoch)
      x, l, p = arm_feature(arm, int(idx), bag)
      l = (l - mu) / sd
      xt = torch.from_numpy(x).to(DEVICE)
      lt = torch.from_numpy(l).to(DEVICE)
      pt = None if p is None else torch.from_numpy(p).to(DEVICE)
      ht = (
          torch.from_numpy(bag_histogram(int(idx))).to(DEVICE)
          if use_hist
          else None
      )
      logit, _ = model(xt, lt, pt, bag_hist=ht)
      target = torch.tensor(float(labels[int(idx)]), device=DEVICE)
      loss = F.binary_cross_entropy_with_logits(logit, target)
      if target.item() == 1:
        loss = loss * pw
      opt.zero_grad()
      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
      opt.step()
      losses.append(float(loss.detach().cpu()))
    vs = predict_mil(
        model, arm, val_idx, mu, sd, use_length=use_length, use_hist=use_hist
    )
    va = (
        float(roc_auc_score(val_y, vs)) if len(np.unique(val_y)) == 2 else 0.5
    )
    history.append(
        dict(epoch=epoch, loss=float(np.mean(losses)), val_auc=va)
    )
    if va > best_auc + 1e-4:
      best_auc = va
      best = copy.deepcopy(model.state_dict())
      patience = 0
    else:
      patience += 1
      if patience >= CFG["PATIENCE"]:
        break
  model.load_state_dict(best)
  payload = dict(
      arm=arm,
      use_length=use_length,
      use_hist=use_hist,
      len_mu=mu,
      len_sd=sd,
      seed=int(seed),
      best_val_auc=float(best_auc),
      state_dict={k: v.detach().cpu() for k, v in model.state_dict().items()},
      history=history,
      protocol_hash=PROTOCOL_HASH,
  )
  if save_path:
    torch.save(payload, save_path)
  return model, payload


@torch.inference_mode()
def predict_mil(
    model,
    arm,
    indices,
    mu,
    sd,
    use_length=True,
    use_hist=True,
    bag=None,
    return_attention=False,
):
  model.eval()
  bag = full_bag() if bag is None else np.asarray(bag, int)
  scores = []
  att = []
  for idx in indices:
    x, l, p = arm_feature(arm, int(idx), bag)
    l = (l - mu) / sd
    xt = torch.from_numpy(x).to(DEVICE)
    lt = torch.from_numpy(l).to(DEVICE)
    pt = None if p is None else torch.from_numpy(p).to(DEVICE)
    ht = (
        torch.from_numpy(bag_histogram(int(idx))).to(DEVICE)
        if use_hist
        else None
    )
    logit, a = model(xt, lt, pt, bag_hist=ht)
    scores.append(float(torch.sigmoid(logit).cpu()))
    if return_attention:
      att.append(a.float().cpu().numpy())
  return (np.array(scores), att) if return_attention else np.array(scores)


def load_mil(path):
  p = torch.load(path, map_location="cpu", weights_only=False)
  if p.get("protocol_hash") != PROTOCOL_HASH:
    raise RuntimeError("Modelo de outro protocolo")
  m = model_for_arm(
      p["arm"],
      use_length=p["use_length"],
      use_hist=p.get("use_hist", CFG["USE_HISTOGRAM"]),
  )
  m.load_state_dict(p["state_dict"])
  m.eval()
  return m, p


print("=" * 90)
print("END CELL 06 — GATED MIL E FUNCOES DE TREINO")
print("=" * 90)
# <<< CELL 06 END =============================================================

In [ ]:
# >>> CELL 07 START — DESENVOLVIMENTO E LOCK NO CRISTIANO =====================
print("\n" + "=" * 90)
print("BEGIN CELL 07 — DESENVOLVIMENTO E LOCK NO CRISTIANO")
print("=" * 90)

MIL_ARMS = ("caduceus", "kmer", "promoter_id", "length_only")
DELFI_ARMS = ("delfi_all", "delfi_prom_all", "delfi_prom_matched")
DELFI_DATA = {
    "delfi_all": DELFI_ALL,
    "delfi_prom_all": DELFI_PROM_ALL,
    "delfi_prom_matched": DELFI_PROM_MATCHED,
}
MODEL_MANIFEST = DIRS["models"] / "MODELS_FROZEN.json"
OOF_FILE = DIRS["models"] / "cristiano_oof_scores.csv"
fold_map = dict(zip(FOLDS.sample_id.astype(str), FOLDS.fold.astype(int)))
SOURCE_FOLD = np.array(
    [fold_map[str(META.loc[i, "sample_id"])] for i in SOURCE_IDX], int
)


def y_for(indices):
  lut = {int(i): int(y) for i, y in zip(SOURCE_IDX, SOURCE_Y)}
  return np.array([lut[int(i)] for i in indices], int)


def make_delfi(seed):
  return Pipeline([
      ("impute", SimpleImputer(strategy="median")),
      ("variance", VarianceThreshold(1e-10)),
      ("scale", StandardScaler()),
      (
          "logit",
          LogisticRegressionCV(
              Cs=np.logspace(-4, 1, 6),
              cv=3,
              scoring="roc_auc",
              penalty="l2",
              solver="liblinear",
              class_weight="balanced",
              max_iter=5000,
              random_state=seed,
              n_jobs=-1,
              refit=True,
          ),
      ),
  ])


oof = {a: np.full(len(SOURCE_IDX), np.nan) for a in (*MIL_ARMS, *DELFI_ARMS)}

for fold in range(CFG["N_FOLDS"]):
  trloc = np.where(SOURCE_FOLD != fold)[0]
  teloc = np.where(SOURCE_FOLD == fold)[0]
  tridx = SOURCE_IDX[trloc]
  teidx = SOURCE_IDX[teloc]
  ytr = SOURCE_Y[trloc]
  itr, iva = source_train_val_split(tridx, ytr, CFG["SEED"] + fold)

  for arm in MIL_ARMS:
    path = DIRS["models"] / f"fold{fold}_{arm}.pt"
    m, p = train_mil(
        arm,
        itr,
        y_for(itr),
        iva,
        y_for(iva),
        CFG["SEED"] + 100 * fold,
        use_length=True,
        use_hist=CFG["USE_HISTOGRAM"],
        save_path=path,
    )
    oof[arm][teloc] = predict_mil(
        m,
        arm,
        teidx,
        p["len_mu"],
        p["len_sd"],
        use_length=True,
        use_hist=CFG["USE_HISTOGRAM"],
    )
    del m
    torch.cuda.empty_cache()

  for arm in DELFI_ARMS:
    pipe = make_delfi(CFG["SEED"] + fold)
    pipe.fit(np.asarray(DELFI_DATA[arm][tridx]), ytr)
    oof[arm][teloc] = pipe.predict_proba(np.asarray(DELFI_DATA[arm][teidx]))[
        :, 1
    ]
    joblib.dump(pipe, DIRS["models"] / f"fold{fold}_{arm}.joblib")

OOF = pd.DataFrame(
    dict(
        sample_id=META.loc[SOURCE_IDX, "sample_id"].astype(str).to_numpy(),
        y=SOURCE_Y,
        fold=SOURCE_FOLD,
        **oof,
    )
)
OOF.to_csv(OOF_FILE, index=False)
print("\nAUROC OOF Cristiano:")
for arm in (*MIL_ARMS, *DELFI_ARMS):
  print(f"  {arm:22s} {roc_auc_score(SOURCE_Y, oof[arm]):.4f}")

for arm in MIL_ARMS:
  for sidx in range(CFG["N_SEEDS"]):
    seed = CFG["SEED"] + 10000 + sidx
    tr, va = source_train_val_split(SOURCE_IDX, SOURCE_Y, seed)
    path = DIRS["models"] / f"final_{arm}_seed{sidx}.pt"
    m, p = train_mil(
        arm,
        tr,
        y_for(tr),
        va,
        y_for(va),
        seed,
        use_length=True,
        use_hist=CFG["USE_HISTOGRAM"],
        save_path=path,
    )
    del m
    torch.cuda.empty_cache()

for arm in DELFI_ARMS:
  pipe = make_delfi(CFG["SEED"] + 10000)
  pipe.fit(np.asarray(DELFI_DATA[arm][SOURCE_IDX]), SOURCE_Y)
  joblib.dump(pipe, DIRS["models"] / f"final_{arm}.joblib")

files = sorted(
    list(DIRS["models"].glob("*.pt")) + list(DIRS["models"].glob("*.joblib"))
)
mm = dict(
    protocol_hash=PROTOCOL_HASH,
    model_hashes={p.name: sha256_file(p) for p in files},
    frozen_at=time.strftime("%Y-%m-%d %H:%M:%S"),
)
MODEL_MANIFEST.write_text(json.dumps(mm, indent=2))
print("Modelos treinados e congelados com sucesso.")

print("=" * 90)
print("END CELL 07 — DESENVOLVIMENTO E LOCK NO CRISTIANO")
print("=" * 90)
# <<< CELL 07 END =============================================================

In [ ]:
# >>> CELL 08 START — AVALIACAO EXTERNA BLOQUEADA NO JIANG ====================
print("\n" + "=" * 90)
print("BEGIN CELL 08 — AVALIACAO EXTERNA BLOQUEADA NO JIANG")
print("=" * 90)

RESULT_MANIFEST = DIRS["results"] / "locked_evaluation_manifest.json"
SCORES_FILE = DIRS["results"] / "jiang_patient_scores.csv"
PRIMARY_FILE = DIRS["results"] / "primary_results.csv"
SECONDARY_FILE = DIRS["results"] / "secondary_results.csv"
target_meta = (
    META.loc[TARGET_IDX, ["sample_id", "condicao"]]
    .copy()
    .reset_index(drop=True)
)
target_cond = target_meta.condicao.astype(str).to_numpy()

scores = {}
for arm in MIL_ARMS:
  per = []
  for sidx in range(CFG["N_SEEDS"]):
    m, p = load_mil(DIRS["models"] / f"final_{arm}_seed{sidx}.pt")
    per.append(
        predict_mil(
            m,
            arm,
            TARGET_IDX,
            p["len_mu"],
            p["len_sd"],
            use_length=True,
            use_hist=CFG["USE_HISTOGRAM"],
        )
    )
    del m
    torch.cuda.empty_cache()
  scores[arm] = np.mean(per, axis=0)

for arm in DELFI_ARMS:
  pipe = joblib.load(DIRS["models"] / f"final_{arm}.joblib")
  scores[arm] = pipe.predict_proba(np.asarray(DELFI_DATA[arm][TARGET_IDX]))[
      :, 1
  ]

score_df = target_meta.copy()
for arm, v in scores.items():
  score_df[arm] = v
score_df.to_csv(SCORES_FILE, index=False)


def paired_bootstrap(y, a, b, name):
  y = np.asarray(y, int)
  a = np.asarray(a, float)
  b = np.asarray(b, float)
  pos = np.where(y == 1)[0]
  neg = np.where(y == 0)[0]
  rng = np.random.default_rng(stable_seed("boot|" + name))
  v = []
  for _ in range(CFG["N_BOOT"]):
    ix = np.r_[rng.choice(pos, len(pos), True), rng.choice(neg, len(neg), True)]
    v.append(roc_auc_score(y[ix], a[ix]) - roc_auc_score(y[ix], b[ix]))
  v = np.asarray(v)
  return v, np.quantile(v, [0.025, 0.975])


def comparison(mask, comp, name):
  y = (target_cond[mask] == CFG["CANCER_LABEL"]).astype(int)
  a = scores["caduceus"][mask]
  b = scores[comp][mask]
  d = delong_paired(y, a, b)
  boots, ci = paired_bootstrap(y, a, b, name + comp)
  d.update(
      dict(
          contrast=name,
          arm_a="caduceus",
          arm_b=comp,
          n_pos=int(y.sum()),
          n_neg=int((y == 0).sum()),
          bootstrap_ci_low=float(ci[0]),
          bootstrap_ci_high=float(ci[1]),
      )
  )
  return d, boots


primary_mask = np.isin(target_cond, [CFG["CANCER_LABEL"], CFG["HEALTHY_LABEL"]])
secondary_mask = np.isin(
    target_cond, [CFG["CANCER_LABEL"], *CFG["LIVER_DISEASE_LABELS"]]
)
comparators = (
    "delfi_all",
    "delfi_prom_all",
    "delfi_prom_matched",
    "kmer",
    "promoter_id",
    "length_only",
)

pr, sr, boot = [], [], {}
for comp in comparators:
  r, b = comparison(primary_mask, comp, "HCC_vs_healthy")
  pr.append(r)
  boot["primary_caduceus_minus_" + comp] = b
  r, b = comparison(secondary_mask, comp, "HCC_vs_hepatitis_cirrhosis")
  sr.append(r)
  boot["secondary_caduceus_minus_" + comp] = b

PRIMARY = pd.DataFrame(pr)
SECONDARY = pd.DataFrame(sr)

main = PRIMARY[PRIMARY.arm_b == "delfi_all"].iloc[0]
if main.bootstrap_ci_high < CFG["DELTA_STAR"]:
  decision = "FUTIL"
elif main.delta >= CFG["DELTA_STAR"] and main.bootstrap_ci_low > 0:
  decision = "SUPERIOR_PROMISSOR"
elif main.delta >= CFG["DELTA_STAR"]:
  decision = "PROMISSOR_REQUER_REPLICACAO"
else:
  decision = "INCONCLUSIVO"

PRIMARY["primary_decision"] = decision
PRIMARY.to_csv(PRIMARY_FILE, index=False)
SECONDARY.to_csv(SECONDARY_FILE, index=False)
np.savez_compressed(DIRS["results"] / "paired_bootstrap.npz", **boot)

auc = []
for name, mask in [
    ("HCC_vs_healthy", primary_mask),
    ("HCC_vs_hepatitis_cirrhosis", secondary_mask),
]:
  y = (target_cond[mask] == CFG["CANCER_LABEL"]).astype(int)
  for arm in (*MIL_ARMS, *DELFI_ARMS):
    auc.append(
        dict(
            contrast=name,
            arm=arm,
            n_pos=int(y.sum()),
            n_neg=int((y == 0).sum()),
            auc=float(roc_auc_score(y, scores[arm][mask])),
        )
    )
AUC = pd.DataFrame(auc)
AUC.to_csv(DIRS["results"] / "auc_by_arm.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {
    "caduceus": "#542788",
    "kmer": "#1b7837",
    "promoter_id": "#d95f02",
    "length_only": "#999999",
    "delfi_all": "#2166ac",
    "delfi_prom_all": "#67a9cf",
    "delfi_prom_matched": "#92c5de",
}
for ax, (name, mask) in zip(
    axes,
    [("HCC vs saudável", primary_mask), ("HCC vs hepatite/cirrose", secondary_mask)],
):
  y = (target_cond[mask] == CFG["CANCER_LABEL"]).astype(int)
  for arm in (*MIL_ARMS, *DELFI_ARMS):
    fpr, tpr, _ = roc_curve(y, scores[arm][mask])
    ax.plot(
        fpr,
        tpr,
        color=colors[arm],
        label=f"{arm} {roc_auc_score(y, scores[arm][mask]):.3f}",
    )
  ax.plot([0, 1], [0, 1], "--", color="grey")
  ax.set_title(name)
  ax.set_xlabel("1-especificidade")
  ax.set_ylabel("sensibilidade")
  ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(DIRS["results"] / "roc_curves.png", dpi=200)
plt.close(fig)

m, p = load_mil(DIRS["models"] / "final_caduceus_seed0.pt")
_, att = predict_mil(
    m,
    "caduceus",
    TARGET_IDX,
    p["len_mu"],
    p["len_sd"],
    use_length=True,
    use_hist=CFG["USE_HISTOGRAM"],
    return_attention=True,
)
del m
torch.cuda.empty_cache()

agg = {}
for local, idx in enumerate(TARGET_IDX):
  group = str(META.loc[idx, "condicao"])
  ids = np.asarray(PROMOTER_IDS[idx])
  weights = np.asarray(att[local])
  for pid, w in zip(ids, weights):
    key = (group, int(pid))
    s, n = agg.get(key, (0.0, 0))
    agg[key] = (s + float(w), n + 1)
att_rows = [
    dict(
        condicao=g,
        promoter_id=pid,
        attention_sum=s,
        n_fragments=n,
        mean_fragment_attention=s / n,
    )
    for (g, pid), (s, n) in agg.items()
]
ATT = pd.DataFrame(att_rows).merge(PLS_DF, on="promoter_id", how="left")
ATT.to_csv(DIRS["results"] / "attention_descriptive_seed0.csv", index=False)

report = []
report.append(
    "# Caduceus Multimodal Híbrido (Promotores PLS): avaliação bloqueada no"
    " Jiang\n"
)
report.append(f"**Decisão primária:** {decision}\n")
report.append("## AUROC\n\n" + AUC.to_markdown(index=False) + "\n")
report.append(
    "## Comparações primárias (vs DELFI e Controles)\n\n"
    + PRIMARY.to_markdown(index=False)
    + "\n"
)
(DIRS["results"] / "final_report.md").write_text("\n".join(report))

print("\nAUROC:")
print(AUC.to_string(index=False))
print("\nComparações primárias:")
print(PRIMARY.to_string(index=False))
print(f"\nDECISÃO: {decision}")

print("=" * 90)
print("END CELL 08 — AVALIACAO EXTERNA BLOQUEADA NO JIANG")
print("=" * 90)
# <<< CELL 08 END =============================================================

In [ ]:
# >>> CELL 09 START — SENSIBILIDADES POS-LOCK (CONDICIONAIS) ==================
print("\n" + "=" * 90)
print("BEGIN CELL 09 — SENSIBILIDADES POS-LOCK")
print("=" * 90)

if not CFG["RUN_POSTLOCK_SENSITIVITIES"]:
  print("Sensibilidades pós-lock desativadas.")
else:
  print("Executando sensibilidade sem comprimento...")

print("=" * 90)
print("END CELL 09 — SENSIBILIDADES POS-LOCK")
print("=" * 90)
# <<< CELL 09 END =============================================================

In [ ]:
# >>> CELL 10 START — MANIFESTO FINAL E PERSISTENCIA ===========================
print("\n" + "=" * 90)
print("BEGIN CELL 10 — MANIFESTO FINAL E PERSISTENCIA")
print("=" * 90)

try:
  G2.close()
except Exception:
  pass
CADUCEUS = None
TOKENIZER = None
torch.cuda.empty_cache()
gc.collect()

all_outputs = []
for d in DIRS.values():
  all_outputs.extend(p for p in d.rglob("*") if p.is_file())
run_manifest = dict(
    study_version=CFG["STUDY_VERSION"],
    notebook_revision=NOTEBOOK_REVISION,
    protocol_hash=PROTOCOL_HASH,
    completed_at=time.strftime("%Y-%m-%d %H:%M:%S"),
    n_outputs=len(all_outputs),
    outputs=[str(p.relative_to(ROOT)) for p in sorted(all_outputs)],
)
(ROOT / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2))

print("EXECUÇÃO CONCLUIDA")
print(f"Resultados : {DIRS['results']}")
print(f"Manifesto  : {ROOT / 'run_manifest.json'}")
print("=" * 90)
print("END CELL 10 — MANIFESTO FINAL E PERSISTENCIA")
print("=" * 90)
# <<< CELL 10 END =============================================================